sec7A — numbers quoted in Sec. VII A (single lossy site) and the caption of Fig. `fig:site`
Every number is re-derived from the shipped data of `fig_site` and printed next to the paper's string; `match`
says whether the new value, rounded as printed, gives the printed string.

In [1]:
USE_OUTPUT = False          # True: rebuild fig_site's data in memory from ../fig_site/output
OUTPUT_DIR = "../fig_site/output"

In [2]:
import sys
import importlib.util
from pathlib import Path
import numpy as np
HERE = Path.cwd()
sys.path.insert(0, str(HERE.parent))
from common import io as IO
if USE_OUTPUT:
    spec = importlib.util.spec_from_file_location("pp", HERE.parent / "fig_site" / "postprocess.py")
    pp = importlib.util.module_from_spec(spec); spec.loader.exec_module(pp)
    S = pp.build(OUTPUT_DIR)
else:
    S = IO.read_data(HERE.parent / "fig_site" / "data")
SM, EN = S["summary.csv"], S["endpoints.csv"]
ROWS = []
def check(where, printed, value, fmt, shown=None):
    s = fmt(value); ROWS.append((where, printed, s, s == (shown or printed)))
sel = lambda m, n0=None: IO.select(SM, method=m, **({} if n0 is None else {"n0": n0}))

In [3]:
d = sel("DSL")
check("DSL max |n - n_ex|, |D - D_ex| (all n0, t)", r"$4\times10^{-8}$", max(d["err_n"].max(), d["err_D"].max()),
      lambda v: f"{v:.0e}", shown="4e-08")
check("2B n_inf (n0-independent), max over n0 of n(gamma t=2000)", "1/4", sel("2B")["n_gt2000"].max(), lambda v: f"{v:.2f}", "0.25")
check("2B n_inf, min over n0", "1/4", sel("2B")["n_gt2000"].min(), lambda v: f"{v:.2f}", "0.25")
check("mGW n_inf, max over n0", "1/2", sel("mGW")["n_gt2000"].max(), lambda v: f"{v:.2f}", "0.50")
check("mGW n_inf, min over n0", "1/2", sel("mGW")["n_gt2000"].min(), lambda v: f"{v:.2f}", "0.50")
check("mGW min D from n0 = 0.9", "$-0.51$", sel("mGW", 0.9)["D_min"][0], lambda v: f"${v:.2f}$")
check("mGW min n_sigma from n0 = 0.9", "$-0.33$", sel("mGW", 0.9)["n_min"][0], lambda v: f"${v:.2f}$")
e = IO.select(EN, method="DSL"); x = IO.select(EN, method="exact"); ex = dict(zip(x["n0"], x["n_up"]))
check("DSL n(2000) = exact n0(1-n0), max deviation", "exact", max(abs(v - ex[n]) for n, v in zip(e["n0"], e["n_up"])),
      lambda v: "exact" if v < 1e-6 else f"{v:.1e}")
# tails (caption: MF ~ 1/(2 gamma t), TPPX ~ (8 gamma t)^(-1/2)): ratios at gamma t = 2000 (-> 1)
for n0 in (0.1, 0.5, 0.9):
    check(f"MF n * 2 gamma t at gamma t = 2000, n0 = {n0}", "~1 (1/(2 gamma t))", sel("HF", n0)["n_gt2000"][0] * 2 * 2000,
          lambda v: f"{v:.2f}", shown=None)
    check(f"TPPX n * sqrt(8 gamma t) at gamma t = 2000, n0 = {n0}", "~1 ((8 gamma t)^-1/2)",
          sel("TPPX", n0)["n_gt2000"][0] * np.sqrt(8 * 2000), lambda v: f"{v:.2f}", shown=None)
check("MF and TPPX keep D >= 0 (min D over all n0)", ">= 0", min(sel("HF")["D_min"].min(), sel("TPPX")["D_min"].min()),
      lambda v: ">= 0" if v > -1e-10 else f"{v:.1e}")

In [4]:
w = max(len(r[0]) for r in ROWS)
print(f"{'quantity':{w}s}  {'paper':22s} {'new':10s} match")
for r in ROWS:
    print(f"{r[0]:{w}s}  {r[1]:22s} {r[2]:10s} {'yes' if r[3] else ('info' if r[1].startswith('~') else 'NO')}")

quantity                                                   paper                  new        match
DSL max |n - n_ex|, |D - D_ex| (all n0, t)                 $4\times10^{-8}$       4e-08      yes
2B n_inf (n0-independent), max over n0 of n(gamma t=2000)  1/4                    0.25       yes
2B n_inf, min over n0                                      1/4                    0.25       yes
mGW n_inf, max over n0                                     1/2                    0.50       yes
mGW n_inf, min over n0                                     1/2                    0.50       yes
mGW min D from n0 = 0.9                                    $-0.51$                $-0.51$    yes
mGW min n_sigma from n0 = 0.9                              $-0.33$                $-0.33$    yes
DSL n(2000) = exact n0(1-n0), max deviation                exact                  exact      yes
MF n * 2 gamma t at gamma t = 2000, n0 = 0.1               ~1 (1/(2 gamma t))     1.00       info
TPPX n * sqrt(8 gamma t) at